# COMP0178 - Lab 6: Cookies and Sessions

Source: [Lab 6 PDF](Lab6.pdf), 22 PDF pages. References below use PDF page numbers. This continues [Lab 5: database and password protection](lab5.ipynb).

**Main idea:** HTTP requests do not automatically remember a login. A browser cookie can carry a session identifier; the server uses that identifier to find trusted session data and checks permissions on every protected request.

- **Tutorial material:** HTTP state, cookies, PHP sessions, login/logout and the project workflow.
- **Corrections and supplementary guidance:** explicitly labelled where older slides simplify security or PHP behaviour. Official PHP/ICO references were checked on 18 November 2026.
- **Practice:** Python 3 and the standard-library `sqlite3` module only. Run code cells in order. These use disposable sample data; PHP fragments are reading examples for a PHP web application, not Python cells.

## 1. Why accounts need state (PDF pages 2-3)

- **HTTP is stateless:** a later request does not inherently prove that its sender authenticated on an earlier request. Dynamic PHP/database content alone does not supply this memory.
- An application needs state for login/logout, private account pages, a shopping basket, recent searches and recently viewed pages.
- Sending `userID=10` in GET or POST does not prove identity: the client can choose that value. The same is true of an ordinary editable user-ID cookie.
- **Authentication:** establish who the user is, for example by verifying a password. **Authorization:** decide whether that user may access a page, record or operation.

Example: Bob may be logged in, but that does not authorize him to edit Alice's auction. A hidden Edit link is convenient UI; the server must still check ownership when the edit request arrives.

## 2. Cookies connect browser requests (PDF pages 4-5)

- A **cookie** is browser-stored data. A response can ask the browser to store it; later eligible requests send it back.
- The tutorial highlights a name, string value and optional expiration date. An explicit future expiry makes it persistent; an expiry in the past requests deletion. Without persistent expiry it is a browser-session cookie.
- Cookies may support a visit or survive across visits. Persistence is a browser-storage choice, not proof of authentication.

Conceptual request/response sequence:

```text
Browser                       Server
POST /login + credentials  ->  Verify credentials
                           <-  Set-Cookie: PHPSESSID=<opaque-id>
GET /account + cookie      ->  Look up <opaque-id> in session store
                           <-  Authorized account page
```

**Supplement:** the slide's "only to the server it came from" is a simplification. Cookie scope also depends on domain/path and security attributes. Cookies are client-controlled input; do not put passwords or trust an unsigned identity claim in them. [PHP cookie options](https://www.php.net/manual/en/function.setcookie.php)

## 3. Setting, reading and expiring a PHP cookie (PDF page 6)

The source introduces `setcookie()` and the `$_COOKIE` superglobal. Use a harmless preference to understand the mechanics:

```php
// PHP 7.3+ options-array syntax. HTTPS application example.
// Send cookies before HTML, echo, or other response output.
setcookie('theme', 'dark', [
    'expires' => time() + 60 * 60 * 24 * 30,
    'path' => '/',
    'secure' => true,
    'httponly' => true,
    'samesite' => 'Lax'
]);

// A subsequent request carries the cookie in $_COOKIE.
$theme = $_COOKIE['theme'] ?? 'light';
if (!in_array($theme, ['light', 'dark'], true)) {
    $theme = 'light';
}

// Request deletion using the same name and cookie scope.
setcookie('theme', '', [
    'expires' => time() - 3600,
    'path' => '/',
    'secure' => true,
    'httponly' => true,
    'samesite' => 'Lax'
]);
```

- Cookie changes are response headers; `$_COOKIE` describes the incoming request, so setting a cookie does not immediately populate it.
- Match the original path/domain when deleting. A cookie scoped to `/account` is distinct from one with the same name scoped to `/`.
- `Secure` restricts transmission to secure connections; `HttpOnly` blocks JavaScript access to the cookie; `SameSite` controls cross-site sending. These attributes complement server checks.

Official syntax and behaviour: [PHP `setcookie()`](https://www.php.net/manual/en/function.setcookie.php). The `secure=true` fragments assume HTTPS; a local HTTP page will not behave like the deployed HTTPS example.

## 4. Read the cookie-login examples critically (PDF pages 7-11)

**What the tutorial is demonstrating:** verify credentials, set cookies, redirect; on another page inspect cookies; expire them to log out. Page 11 correctly warns that clients can view and fake cookies.

**Corrections to the examples:**

- Pages 7-9 treat `userID`/`username` cookie presence as a login check. That is insufficient: changing a cookie must not let someone become another user. Use a server-managed authenticated session.
- Page 8 combines user input into SQL and compares a `SHA()` password digest. Reuse Lab 5's prepared-statement and password-hashing approach: select by username with a bound parameter, then verify the stored password hash with `password_verify()`.
- Cookie data and usernames remain untrusted when displayed; escape them for HTML with `htmlspecialchars()`.
- Page 10 constructs `$indexURL` but redirects to `$home_url`, an inconsistent variable. A fixed application-relative destination such as `/index.php` avoids this typo; call `exit` after redirecting.
- Deleting ordinary identity cookies does not revoke a separate server session. Logout must invalidate the authentication state that the application actually uses.

Official references: [MySQLi prepared statements](https://www.php.net/manual/en/mysqli.quickstart.prepared-statements.php), [PHP `password_verify()`](https://www.php.net/manual/en/function.password-verify.php). SQL parameter binding and password verification solve different problems; both are needed.

## 5. Cookies and privacy (PDF page 12)

- The slide introduces UK PECR and the idea of explaining cookie purposes and obtaining consent where required. It notes an essential-cookie exception and that similar device-storage technologies can also fall within the rules.
- **Study distinction:** a cookie used for a requested login or basket has a different purpose from optional advertising/tracking. The word "cookie" alone does not determine the consent analysis.
- Do not read the slide as a complete current legal checklist. The [ICO cookies overview](https://ico.org.uk/for-organisations/direct-marketing-and-privacy-and-electronic-communications/guide-to-pecr/cookies-and-similar-technologies/) states that it is based on previous detailed guidance and links to revised guidance under consultation. Use the current ICO guidance when assessing an actual site.

The practical revision point is to identify each storage purpose, explain it clearly and determine which requirements/exceptions apply, rather than copying a generic banner.

## 6. What a PHP session stores (PDF pages 13-16)

| Concept | Browser/client | Server |
|---|---|---|
| Ordinary cookie | Stores a string and sends it on eligible requests | Reads incoming cookie values |
| PHP session | Usually stores an opaque session-ID cookie | Stores data associated with that ID, exposed through `$_SESSION` |
| Authenticated session | Possesses an identifier | Records the verified user identity and checks validity/permissions |

- `session_start()` starts a session or resumes one associated with the incoming ID. It must run before accessing session data on each relevant PHP request.
- `$_SESSION['userID']` is server-side application data. The browser normally receives the session ID, rather than the whole session array.
- An ID is a bearer secret: someone who steals a valid authenticated ID may be able to use the session. Server storage does not make ID theft harmless.

**Two corrections to the slides:**

1. Page 16 reverses cookie name/value. The usual cookie is `PHPSESSID=<session-id>`: a configured session **name** identifies the cookie and its **value** contains the ID. [PHP configuration](https://www.php.net/manual/en/session.configuration.php)
2. Pages 14-15 say sessions automatically end when the browser closes. Browser cookie lifetime and server-side state lifetime are separate. Server data can remain until expiry/cleanup, and closing the browser is not a reliable server logout signal. Enforce server-side timeouts and explicit logout. [PHP session configuration](https://www.php.net/manual/en/session.configuration.php)

The source also describes URL parameters as an alternative ID transport. PHP supports historical URL propagation, but its security guidance prefers cookies: URLs can leak IDs. Enabling transparent URL-ID propagation is deprecated from PHP 8.4. [Passing session IDs](https://www.php.net/manual/en/session.idpassing.php), [configuration/version notes](https://www.php.net/manual/en/session.configuration.php)

## 7. Session startup and login (PDF pages 15, 17)

**Supplementary PHP reading example:** put shared session configuration in application bootstrap code, before output and before `session_start()`. Options-array syntax here requires PHP 7.3+.

```php
ini_set('session.use_strict_mode', '1');
ini_set('session.use_only_cookies', '1');
session_set_cookie_params([
    'lifetime' => 0,
    'path' => '/',
    'secure' => true,
    'httponly' => true,
    'samesite' => 'Lax'
]);
session_start();
```

Set these parameters on each request before startup. See [PHP `session_set_cookie_params()`](https://www.php.net/manual/en/function.session-set-cookie-params.php) and [session security settings](https://www.php.net/manual/en/session.security.ini.php).

Login flow, assuming an anonymous session is already started:

```php
// $connection, $username and $submittedPassword are application inputs.
// Registration stored a password_hash() result in password_hash.
$stmt = $connection->prepare(
    'SELECT userID, username, password_hash FROM User WHERE username = ?'
);
$stmt->bind_param('s', $username);
$stmt->execute();
$stmt->bind_result($userID, $dbUsername, $storedHash);
$found = $stmt->fetch();
$stmt->close();

if ($found && password_verify($submittedPassword, $storedHash)) {
    // Regenerate before adding authenticated identity to the new session.
    session_regenerate_id();
    $_SESSION['userID'] = $userID;
    $_SESSION['username'] = $dbUsername;
    header('Location: /index.php');
    exit;
}
echo 'Invalid username or password';
```

- This is a flow fragment, not a full login endpoint. It assumes a unique username and a correctly initialized MySQLi connection/schema.
- Regenerate the ID when authentication raises privileges. A full application needs the documented handling of obsolete IDs, timeouts and concurrent requests; regeneration is not the entire session policy. [PHP session management](https://www.php.net/manual/en/features.session.security.management.php)
- Page 17 adds persistent `userID`/`username` cookies alongside a session. Those values must not become independent authentication credentials. A secure "remember me" design requires separate revocable authentication tokens; simply remembering an ID is insufficient.

## 8. Protected pages and displaying login status (PDF pages 18-19)

```php
// The shared bootstrap has configured and started the session.
if (!isset($_SESSION['userID'])) {
    header('Location: /login.php');
    exit;
}

$userID = (int) $_SESSION['userID']; // Identity from server session.
echo 'You are logged in as ' . htmlspecialchars(
    $_SESSION['username'], ENT_QUOTES | ENT_SUBSTITUTE, 'UTF-8'
);
```

- The tutorial's conditional Login/Register/Logout links reflect session state. They improve the interface; endpoint checks provide access control.
- For account data, bind the authenticated server-side `userID` in the SQL predicate. Do not replace it with a requested `?userID=...` value.
- For editing/deleting an auction, check that the authenticated user owns that auction (or has an authorized role). Being logged in alone is not ownership.
- For state-changing requests, include appropriate CSRF protection; cookies may be sent automatically with a request. [PHP session security discussion](https://www.php.net/manual/en/features.session.security.management.php)

**Distinction:** a PHP/HTTP session tracks state across web requests. A **database transaction** groups database operations with ACID properties. A person can have one login session containing many separate database transactions; see [Lecture 6 notes](../notes/lesson6.ipynb).

## 9. Logging out (PDF pages 10, 16, 20)

The source combines three actions: clear session variables, expire the session cookie, and destroy session data. Extra ordinary cookies should also be expired if the application created them.

```php
// Basic PHP teaching flow; run before output.
session_start();
$_SESSION = [];

$p = session_get_cookie_params();
setcookie(session_name(), '', [
    'expires' => time() - 3600,
    'path' => $p['path'],
    'domain' => $p['domain'],
    'secure' => $p['secure'],
    'httponly' => $p['httponly'],
    'samesite' => $p['samesite'] ?? 'Lax'
]);
session_destroy();
header('Location: /index.php');
exit;
```

**Correction/supplement:** `session_destroy()` alone does not clear `$_SESSION` in the running script or remove the browser cookie. The manual also warns about immediate deletion with concurrent requests: production logout must reliably revoke access and coordinate session invalidation. [PHP `session_destroy()`](https://www.php.net/manual/en/function.session-destroy.php)

Deleting a browser cookie is not a substitute for revoking server-side authentication; a copied old credential must stop authorizing requests.

## 10. Runnable practice: trusted identity and database access

This is a **supplementary model**, not PHP's session implementation. Python keeps a server-side session dictionary, SQLite holds toy users/auctions/bids, and fixed times make the checks repeatable. Issuing a session below stands in for a **successful prior credential check**; this exercise does not implement password authentication.

- Sample users: Alice (`1`) and Bob (`2`). Bob sells a camera; Alice sells a laptop.
- The browser supplies an opaque session token. The server finds the identity in its own dictionary.
- Sessions expire at a server-side deadline and can be revoked by logout.
- No files or real account data are changed. Run the setup cell again to reset the exercise.

In [1]:
import sqlite3
import secrets

db = sqlite3.connect(':memory:')
db.executescript('''
PRAGMA foreign_keys = ON;
CREATE TABLE User (
    user_id INTEGER PRIMARY KEY,
    username TEXT NOT NULL UNIQUE
);
CREATE TABLE Listing (
    listing_id INTEGER PRIMARY KEY,
    seller_id INTEGER NOT NULL REFERENCES User(user_id),
    title TEXT NOT NULL
);
CREATE TABLE Bid (
    bid_id INTEGER PRIMARY KEY,
    listing_id INTEGER NOT NULL REFERENCES Listing(listing_id),
    bidder_id INTEGER NOT NULL REFERENCES User(user_id),
    amount_cents INTEGER NOT NULL CHECK (amount_cents > 0)
);
INSERT INTO User VALUES (1, 'alice'), (2, 'bob');
INSERT INTO Listing VALUES (10, 2, 'Camera'), (11, 1, 'Laptop');
INSERT INTO Bid VALUES (100, 10, 1, 25000), (101, 11, 2, 30000),
                       (102, 10, 1, 26000);
''')

server_sessions = {}

def issue_after_verified_login(user_id, now=1000):
    # Precondition: an external credential check already verified user_id.
    token = secrets.token_urlsafe(32)
    server_sessions[token] = {'user_id': user_id, 'expires_at': now + 300}
    return token

def authenticated_user(token, now=1000):
    session = server_sessions.get(token)
    if session is None or now >= session['expires_at']:
        raise PermissionError('Login required')
    return session['user_id']

alice_token = issue_after_verified_login(1)
bob_token = issue_after_verified_login(2)
print('Sample database and two post-login sessions ready.')

Sample database and two post-login sessions ready.


### Exercise A - explain the trust boundary

An authenticated Alice request includes a valid Alice session token **and** a forged `userID=2` cookie. Which value should determine whose bids the application returns, and why? Write your answer in the cell, then compare with the explanation below.

In [2]:
# Your conceptual answer (not sent anywhere):
trust_boundary_answer = ""

<details><summary>Answer A</summary>

Use the user ID resolved from the valid server-side session: Alice is user 1. The client-controlled userID cookie is not evidence of Bob's identity. Parameterized SQL still needs the correct trusted identity; binding a forged user ID would prevent SQL injection but would not fix broken authorization.
</details>

In [3]:
def bids_for_request(cookies, now=1000):
    user_id = authenticated_user(cookies.get('sid'), now)
    # cookies['userID'] is deliberately not an authentication source.
    return db.execute('''
        SELECT l.title, seller.username, b.amount_cents
        FROM Bid AS b
        JOIN Listing AS l ON l.listing_id = b.listing_id
        JOIN User AS seller ON seller.user_id = l.seller_id
        WHERE b.bidder_id = ?
        ORDER BY b.bid_id
    ''', (user_id,)).fetchall()

request = {'sid': alice_token, 'userID': '2'}
alice_bids = bids_for_request(request)
assert alice_bids == [('Camera', 'bob', 25000), ('Camera', 'bob', 26000)]
print(alice_bids)
print('Forged userID cookie did not change the authenticated identity.')

[('Camera', 'bob', 25000), ('Camera', 'bob', 26000)]
Forged userID cookie did not change the authenticated identity.


### Exercise B - SQL and ownership checks

1. Write a parameterized query that lists the authenticated user's own auctions, ordered by `listing_id`.
2. Alice tries to edit Camera (`10`). Bob tries the same edit. Who should be permitted?

Use the cell for your query or reasoning; the next cell provides a checked solution. SQLite's `?` parameters below are bound by Python's `execute`, while PHP/MySQLi uses `bind_param()`. This is a conceptual/dialect bridge, not a MySQL server connection.

In [4]:
# Optional scratch work. The query must use the server-resolved user ID.
my_ownership_query = ""
ownership_answer = ""

In [5]:
def own_listings(token):
    user_id = authenticated_user(token)
    return db.execute(
        'SELECT listing_id, title FROM Listing WHERE seller_id = ? '
        'ORDER BY listing_id', (user_id,)
    ).fetchall()

def may_edit_listing(token, listing_id):
    user_id = authenticated_user(token)
    return db.execute(
        'SELECT 1 FROM Listing WHERE listing_id = ? AND seller_id = ?',
        (listing_id, user_id)
    ).fetchone() is not None

assert own_listings(alice_token) == [(11, 'Laptop')]
assert not may_edit_listing(alice_token, 10)
assert may_edit_listing(bob_token, 10)
print('Alice owns:', own_listings(alice_token))
print('Camera edit: Alice denied; Bob permitted.')

Alice owns: [(11, 'Laptop')]
Camera edit: Alice denied; Bob permitted.


### Exercise C - input values, expiry and logout

- Predict the result of looking up a username whose value contains SQL syntax when it is passed as a bound parameter.
- Predict whether a session expiring at time `1300` works at exactly `1300` in this model.
- Predict whether copying the old token before logout should let it continue working afterwards.

In [6]:
attack_like_username = "alice' OR 1=1 --"
rows = db.execute(
    'SELECT user_id FROM User WHERE username = ?', (attack_like_username,)
).fetchall()
assert rows == []
print('Bound input remains a literal username:', rows)

def denied(call):
    try:
        call()
    except PermissionError:
        return True
    return False

assert denied(lambda: authenticated_user(alice_token, now=1300))
assert denied(lambda: authenticated_user('invented-token'))

copied_token = alice_token
server_sessions.pop(alice_token, None)  # Server-side revocation at logout.
assert denied(lambda: authenticated_user(copied_token))
print('Unknown, expired and logged-out tokens are rejected.')
db.close()

Bound input remains a literal username: []
Unknown, expired and logged-out tokens are rejected.


### What the model demonstrates, and what it abstracts away

- **Demonstrates:** trusted server identity, parameterized database access, per-record authorization, expiry and revocation.
- **Abstracts away:** HTTP/browser cookie transport, PHP session storage, password checking, TLS, CSRF, concurrency, persistence, token rotation and cleanup.
- A process-local Python dictionary is only a teaching store; it is not a deployment recommendation. Its session creation function assumes authentication has already happened.

## 11. Revision questions with answers

<details><summary>1. Why does a private page need a check on every request?</summary>

HTTP does not remember authentication on its own. Every protected request must resolve a valid authenticated session and check that user's permission for the requested resource.
</details>

<details><summary>2. Does a userID cookie prove the request came from that user?</summary>

No. The client can forge it. A valid session identifier refers to server-managed authenticated state; an arbitrary identity claim is not enough.
</details>

<details><summary>3. Where are PHP session data and the session identifier normally stored?</summary>

Session data are server-side. The browser normally stores a cookie whose name is PHP's configured session name (commonly PHPSESSID), and whose value is the opaque session ID.
</details>

<details><summary>4. Does session_destroy() alone delete the cookie and clear the current $_SESSION array?</summary>

No. Those are separate cleanup actions. Logout must revoke authenticated access on the server; cookie expiry also removes the browser's stored identifier when accepted.
</details>

<details><summary>5. Why is browser close an unreliable logout policy?</summary>

Browser cookie lifetime and server-side session lifetime are separate. The server needs explicit validity checks, timeouts and logout/revocation, rather than assuming it receives a browser-close notification.
</details>

<details><summary>6. Why regenerate a session ID after successful login?</summary>

Authentication changes privileges. Regenerating helps prevent a pre-login identifier from becoming a reusable authenticated credential; follow the documented procedure for obsolete IDs and concurrent requests.
</details>

<details><summary>7. Why can a prepared statement still return another user's data in a broken application?</summary>

Parameters prevent input from becoming SQL syntax. If the application binds an untrusted requested user ID instead of the server-authenticated ID, its authorization can still be wrong.
</details>

<details><summary>8. Is hiding a Delete button sufficient access control?</summary>

No. A user can send the request directly. The endpoint must verify login, resource ownership/role, and protections appropriate to the state-changing request.
</details>

<details><summary>9. What is the difference between a login session and a database transaction?</summary>

A login session carries authenticated state across requests. A database transaction groups database operations into an ACID unit; many transactions can occur during one login session.
</details>

## 12. Project connection and lab checklist (PDF pages 21-22)

The source ends by linking cookies/sessions to user accounts and listing project work: **registration, auction creation, search and bidding**. Those are tutorial tasks to study and plan; these notes do not change the project application.

- **Registration:** validate data and store a password hash (Lab 5).
- **Login:** verify credentials, establish authenticated server session state and redirect.
- **Auction creation:** use the authenticated user as the seller; validate submitted auction data.
- **Search:** decide which results are public and which require authorization.
- **Bidding:** resolve the bidder from the session, check auction/business rules, and use appropriate database transactions for concurrent changes (Lecture 6).
- **Logout:** revoke server authentication and clean up the relevant browser cookie.

This tutorial is principally project/web-application practice. It does not provide a past-paper cookie/session question, so no exam question number is assigned here. Use [the exam guide](../exam_guide.ipynb) for the reviewed past-paper mapping and [Lecture 6 notes](../notes/lesson6.ipynb) for database transaction revision.

- [ ] I can explain HTTP statelessness and why GET/POST/cookie user IDs are not authentication.
- [ ] I can trace setting, sending, reading and expiring a cookie.
- [ ] I know the difference between the session cookie and server session data.
- [ ] I can distinguish browser-cookie expiry, server timeouts and explicit logout.
- [ ] I can explain authentication, authorization, SQL injection prevention and output escaping separately.
- [ ] I can trace Alice/Bob ownership checks and identify which user ID is trusted.
- [ ] I can identify and correct the old examples on PDF pages 8-10 and 14-17.